In [1]:
import numpy as np
import pandas as pd
from linearmodels.panel import PanelOLS, RandomEffects, PooledOLS
from scipy.stats import chi2, f
from panel_utils import ModelResultsAggregator, run_panel_regressions, run_spec_tests


In [ ]:
###############################
# ЗАГРУЗКА И ПОДГОТОВКА ДАННЫХ
###############################
df_reg_analys = pd.read_excel('reg_analys.xlsx')
df_fed_analys = pd.read_excel('fed_analys.xlsx')


# Убираем служебные столбцы индекса
df_reg_analys = df_reg_analys.loc[:, ~df_reg_analys.columns.str.startswith('Unnamed')]
df_fed_analys = df_fed_analys.loc[:, ~df_fed_analys.columns.str.startswith('Unnamed')]

# Приводим даты к datetime
df_reg_analys['Date'] = pd.to_datetime(df_reg_analys['Date'])
df_fed_analys['Date'] = pd.to_datetime(df_fed_analys['Date'])

# Разделим Mon_Shock на позитивный и негативный
df_reg_analys['Mon_Shock_neg'] = df_reg_analys['Mon_Shock'].where(df_reg_analys['Mon_Shock'] < 0, 0)
df_reg_analys['Mon_Shock_pos'] = df_reg_analys['Mon_Shock'].where(df_reg_analys['Mon_Shock'] > 0, 0)

# Объединяем региональные и федеральные данные
fed_extra_cols = [
    col for col in df_fed_analys.columns
    if col not in df_reg_analys.columns and col != 'Region'
]
df_reg = df_reg_analys.merge(
    df_fed_analys[['Date'] + fed_extra_cols],
    on='Date',
    how='left'
)


# Взаимодействия (если требуется)
if 'Mon_Shock' in df_reg.columns and 'Cluster_1' in df_reg.columns:
    df_reg['Mon_Shock_Cl1'] = df_reg['Mon_Shock'] * df_reg['Cluster_1']
if 'Mon_Shock' in df_reg.columns and 'Cluster_2' in df_reg.columns:
    df_reg['Mon_Shock_Cl2'] = df_reg['Mon_Shock'] * df_reg['Cluster_2']
if 'Mon_Shock' in df_reg.columns and 'Covid_dum' in df_reg.columns:
    df_reg['Mon_Shock_Covid'] = df_reg['Mon_Shock'] * df_reg['Covid_dum']

# Кластерные выборки
if 'Cluster_1' in df_reg.columns and 'Cluster_2' in df_reg.columns:
    df_reg_clus_one = df_reg[df_reg['Cluster_1'] == 1].copy()
    df_reg_clus_two = df_reg[df_reg['Cluster_2'] == 1].copy()
    df_reg_clus_three = df_reg[(df_reg['Cluster_1'] == 0) & (df_reg['Cluster_2'] == 0)].copy()


In [20]:
# # quick column comparison
# reg_cols = df_reg_analys.columns.tolist()
# fed_cols = df_fed_analys.columns.tolist()

# cols_common = [c for c in reg_cols if c in fed_cols]
# cols_only_reg = [c for c in reg_cols if c not in fed_cols]
# cols_only_fed = [c for c in fed_cols if c not in reg_cols]

# cols_common, cols_only_reg, cols_only_fed


In [ ]:
###############################
# Построение линейных моделей на панельных данных (общая выборка)
# (d_Int_Rate_ConsCred зависимая переменная)
###############################
print("="*70)
print("ПАНЕЛЬНАЯ РЕГРЕССИЯ: POOL + FE + RE (ОБЩАЯ ВЫБОРКА)")
print("Зависимая переменная: d_Int_Rate_ConsCred_adj")
print("="*70)

exog_vars_initial = [
    'd_Int_Rate_ConsCred_lag1_adj',
    'd_Cred_nagr_adj',
    'd_D_top5_rozn_adj',
    'd_Fin_Dostup_adj',
    'Credit_impulse_adj',
    'd_Cred_structure_adj',
    'd_Def_Zadolg_ConsCred_adj',
    'Mon_Shock',
    'd_ln_New_Loans_ConsCred_adj',
    'd_Bonds_Rate_Correct_5Y_adj',
    'd_Exc_rate_adj',
    'Inflation_Expectations_adj',
    'Covid_dum',
    'Sank_dum'
]

dependent_var = 'd_Int_Rate_ConsCred_adj'

# Создаем копию df_reg для работы
df_clean = df_reg.copy()  # Ð»Ð¾Ð³Ð¸ÐºÐ° Ð±ÐµÐ· Ð¸Ð·Ð¼ÐµÐ½ÐµÐ½Ð¸Ð¹

# Приводим df_reg к формату с Region и Date как столбцами
if isinstance(df_reg.index, pd.MultiIndex):
    df_reg_temp = df_reg.reset_index()
else:
    df_reg_temp = df_reg.copy()

# Проверяем наличие зависимой переменной

if dependent_var in df_reg_temp.columns:
    
    # Присоединяем зависимую переменную по Region и Date
    df_clean = df_clean.merge(
        df_reg_temp[['Region', 'Date', dependent_var]],
        on=['Region', 'Date'],
        how='left',
        suffixes=('', '_from_reg')
    )
else:
    print(f"ERROR - {dependent_var} отсутствует в df_reg_temp")

# Проверяем независимые переменные
exog_vars_for_regression = []
for var in exog_vars_initial:
    if var in df_clean.columns:
        exog_vars_for_regression.append(var)
    else:
        print(f"  WARNING: {var} исключена")

# Удаляем NaN
cols_to_check = [dependent_var] + exog_vars_for_regression
df_clean = df_clean.dropna(subset=cols_to_check)


# ===== Установка панельного индекса =====
df_clean = df_clean.set_index(['Region', 'Date']).sort_index()

# ===== Подготовка Y и X =====
y = df_clean[[dependent_var]]
X = df_clean[exog_vars_for_regression]


# ===== POOLED OLS =====
pooled_res, fe_res, re_res, pooled_success, fe_success, re_success = run_panel_regressions(
    y, X, cov_type='clustered', cluster_entity=True
)


In [ ]:
# Сохраняем результаты модели для экспорта
pooled_res_1 = pooled_res if pooled_success else None
fe_res_1 = fe_res if fe_success else None
re_res_1 = re_res if re_success else None


In [ ]:
# ===== ТЕСТЫ СПЕЦИФИКАЦИИ =====
print("\n" + "="*70)
print("ТЕСТЫ СПЕЦИФИКАЦИИ")
print("="*70)


run_spec_tests(
    y, X, pooled_res, fe_res, re_res, pooled_success, fe_success, re_success
)


In [ ]:
# Добавляем результаты в сводную таблицу
aggregator_Int_ConsCred = ModelResultsAggregator()

aggregator_Int_ConsCred.add_model_results(
    fe_res,
    dependent_variable='d_Int_Rate_ConsCred_adj',
    subsample_name='Общая выборка',
    model_type='FE',
    specification_name='Модель_4',
    se_type = 'Clustered'
)

In [ ]:
###############################
# Построение линейных моделей на панельных данных (общая выборка)
# (d_Int_Rate_ConsCred зависимая переменная)
###############################
print("="*70)
print("ПАНЕЛЬНАЯ РЕГРЕССИЯ: POOL + FE + RE (ОБЩАЯ ВЫБОРКА)")
print("Зависимая переменная: d_Int_Rate_ConsCred_adj")
print("="*70)

exog_vars_initial = [
    'd_Int_Rate_ConsCred_lag1_adj',
    'd_Cred_nagr_adj',
    'd_D_top5_rozn_adj',
    'd_Fin_Dostup_adj',
    'Credit_impulse_adj',
    'd_Cred_structure_adj',
    'd_Def_Zadolg_ConsCred_adj',
    'Mon_Shock',
    'd_ln_New_Loans_ConsCred_adj',
    'd_Bonds_Rate_Correct_5Y_adj',
    'd_Exc_rate_adj',
    'Inflation_Expectations_adj',
    'Covid_dum',
    'Sank_dum',
    'Mon_Shock_Covid'
]

dependent_var = 'd_Int_Rate_ConsCred_adj'

# Создаем копию df_reg для работы
df_clean = df_reg.copy()  # Ð»Ð¾Ð³Ð¸ÐºÐ° Ð±ÐµÐ· Ð¸Ð·Ð¼ÐµÐ½ÐµÐ½Ð¸Ð¹

# Приводим df_reg к формату с Region и Date как столбцами
if isinstance(df_reg.index, pd.MultiIndex):
    df_reg_temp = df_reg.reset_index()
else:
    df_reg_temp = df_reg.copy()

# Проверяем наличие зависимой переменной

if dependent_var in df_reg_temp.columns:
    
    # Присоединяем зависимую переменную по Region и Date
    df_clean = df_clean.merge(
        df_reg_temp[['Region', 'Date', dependent_var]],
        on=['Region', 'Date'],
        how='left',
        suffixes=('', '_from_reg')
    )
else:
    print(f"ERROR - {dependent_var} отсутствует в df_reg_temp")

# Проверяем независимые переменные
exog_vars_for_regression = []
for var in exog_vars_initial:
    if var in df_clean.columns:
        exog_vars_for_regression.append(var)
    else:
        print(f"  WARNING: {var} исключена")

# Удаляем NaN
cols_to_check = [dependent_var] + exog_vars_for_regression
df_clean = df_clean.dropna(subset=cols_to_check)


# ===== Установка панельного индекса =====
df_clean = df_clean.set_index(['Region', 'Date']).sort_index()

# ===== Подготовка Y и X =====
y = df_clean[[dependent_var]]
X = df_clean[exog_vars_for_regression]


# ===== POOLED OLS =====
pooled_res, fe_res, re_res, pooled_success, fe_success, re_success = run_panel_regressions(
    y, X, cov_type='clustered', cluster_entity=True
)


In [ ]:
# Сохраняем результаты модели для экспорта
pooled_res_2 = pooled_res if pooled_success else None
fe_res_2 = fe_res if fe_success else None
re_res_2 = re_res if re_success else None


In [ ]:
# ===== ТЕСТЫ СПЕЦИФИКАЦИИ =====
print("\n" + "="*70)
print("ТЕСТЫ СПЕЦИФИКАЦИИ")
print("="*70)


run_spec_tests(
    y, X, pooled_res, fe_res, re_res, pooled_success, fe_success, re_success
)


In [ ]:
# Добавляем результаты в сводную таблицу
aggregator_Int_ConsCred.add_model_results(
    fe_res,
    dependent_variable='d_Int_Rate_ConsCred_adj',
    subsample_name='Общая выборка',
    model_type='FE',
    specification_name='Модель_5',
    se_type = 'Clustered'
)

In [ ]:
###############################
# Построение линейных моделей на панельных данных (общая выборка)
# (d_Int_Rate_ConsCred зависимая переменная)
###############################
print("="*70)
print("ПАНЕЛЬНАЯ РЕГРЕССИЯ: POOL + FE + RE (ОБЩАЯ ВЫБОРКА)")
print("Зависимая переменная: d_Int_Rate_ConsCred_adj")
print("="*70)

exog_vars_initial = [
    'd_Int_Rate_ConsCred_lag1_adj',
    'd_Cred_nagr_adj',
    'd_D_top5_rozn_adj',
    'd_Fin_Dostup_adj',
    'Credit_impulse_adj',
    'd_Cred_structure_adj',
    'd_Def_Zadolg_ConsCred_adj',
    'Mon_Shock',
    'd_ln_New_Loans_ConsCred_adj',
    'd_Bonds_Rate_Correct_5Y_adj',
    'd_Exc_rate_adj',
    'Inflation_Expectations_adj',
    'Covid_dum',
    'Sank_dum',
    'Mon_Shock_Cl1',
    'Mon_Shock_Cl2'
]

dependent_var = 'd_Int_Rate_ConsCred_adj'

# Создаем копию df_reg для работы
df_clean = df_reg.copy()  # Ð»Ð¾Ð³Ð¸ÐºÐ° Ð±ÐµÐ· Ð¸Ð·Ð¼ÐµÐ½ÐµÐ½Ð¸Ð¹

# Приводим df_reg к формату с Region и Date как столбцами
if isinstance(df_reg.index, pd.MultiIndex):
    df_reg_temp = df_reg.reset_index()
else:
    df_reg_temp = df_reg.copy()

# Проверяем наличие зависимой переменной

if dependent_var in df_reg_temp.columns:
    
    # Присоединяем зависимую переменную по Region и Date
    df_clean = df_clean.merge(
        df_reg_temp[['Region', 'Date', dependent_var]],
        on=['Region', 'Date'],
        how='left',
        suffixes=('', '_from_reg')
    )
else:
    print(f"ERROR - {dependent_var} отсутствует в df_reg_temp")

# Проверяем независимые переменные
exog_vars_for_regression = []
for var in exog_vars_initial:
    if var in df_clean.columns:
        exog_vars_for_regression.append(var)
    else:
        print(f"  WARNING: {var} исключена")

# Удаляем NaN
cols_to_check = [dependent_var] + exog_vars_for_regression
df_clean = df_clean.dropna(subset=cols_to_check)


# ===== Установка панельного индекса =====
df_clean = df_clean.set_index(['Region', 'Date']).sort_index()

# ===== Подготовка Y и X =====
y = df_clean[[dependent_var]]
X = df_clean[exog_vars_for_regression]


# ===== POOLED OLS =====
pooled_res, fe_res, re_res, pooled_success, fe_success, re_success = run_panel_regressions(
    y, X, cov_type='clustered', cluster_entity=True
)


In [ ]:
# Сохраняем результаты модели для экспорта
pooled_res_3 = pooled_res if pooled_success else None
fe_res_3 = fe_res if fe_success else None
re_res_3 = re_res if re_success else None


In [ ]:
# ===== ТЕСТЫ СПЕЦИФИКАЦИИ =====
print("\n" + "="*70)
print("ТЕСТЫ СПЕЦИФИКАЦИИ")
print("="*70)


run_spec_tests(
    y, X, pooled_res, fe_res, re_res, pooled_success, fe_success, re_success
)


In [ ]:
# Добавляем результаты в сводную таблицу
aggregator_Int_ConsCred.add_model_results(
    fe_res,
    dependent_variable='d_Int_Rate_ConsCred_adj',
    subsample_name='Общая выборка',
    model_type='FE',
    specification_name='Модель_6',
    se_type = 'Clustered'
)

In [ ]:
###############################
# Построение линейных моделей на панельных данных (кластер 1)
# (d_Int_Rate_ConsCred зависимая переменная)
###############################
print("="*70)
print("ПАНЕЛЬНАЯ РЕГРЕССИЯ: POOL + FE + RE (КЛАСТЕР 1)")
print("Зависимая переменная: d_Int_Rate_ConsCred_adj")
print("="*70)

exog_vars_initial = [
    'd_Int_Rate_ConsCred_lag1_adj',
    'd_Cred_nagr_adj',
    'd_D_top5_rozn_adj',
    'd_Fin_Dostup_adj',
    'Credit_impulse_adj',
    'd_Cred_structure_adj',
    'd_Def_Zadolg_ConsCred_adj',
    #'d_ln_New_Loans_Progr',
    'Mon_Shock',
    'd_ln_New_Loans_ConsCred_adj',
    'd_Bonds_Rate_Correct_5Y_adj',
    'd_Exc_rate_adj',
    'Inflation_Expectations_adj'                          
]

dependent_var = 'd_Int_Rate_ConsCred_adj'

# Создаем копию df_reg для работы
df_clean = df_reg_clus_one.copy()  # Ð»Ð¾Ð³Ð¸ÐºÐ° Ð±ÐµÐ· Ð¸Ð·Ð¼ÐµÐ½ÐµÐ½Ð¸Ð¹

# Приводим df_reg к формату с Region и Date как столбцами
if isinstance(df_reg.index, pd.MultiIndex):
    df_reg_temp = df_reg.reset_index()
else:
    df_reg_temp = df_reg.copy()

# Проверяем наличие зависимой переменной

if dependent_var in df_reg_temp.columns:
    
    # Присоединяем зависимую переменную по Region и Date
    df_clean = df_clean.merge(
        df_reg_temp[['Region', 'Date', dependent_var]],
        on=['Region', 'Date'],
        how='left',
        suffixes=('', '_from_reg')
    )
else:
    print(f"ERROR - {dependent_var} отсутствует в df_reg_temp")

# Проверяем независимые переменные
exog_vars_for_regression = []
for var in exog_vars_initial:
    if var in df_clean.columns:
        exog_vars_for_regression.append(var)
    else:
        print(f"  WARNING: {var} исключена")

# Удаляем NaN
cols_to_check = [dependent_var] + exog_vars_for_regression
df_clean = df_clean.dropna(subset=cols_to_check)


# ===== Установка панельного индекса =====
df_clean = df_clean.set_index(['Region', 'Date']).sort_index()

# ===== Подготовка Y и X =====
y = df_clean[[dependent_var]]
X = df_clean[exog_vars_for_regression]


# ===== POOLED OLS =====
pooled_res, fe_res, re_res, pooled_success, fe_success, re_success = run_panel_regressions(
    y, X, cov_type='robust', cluster_entity=None
)


In [ ]:
# Сохраняем результаты модели для экспорта
pooled_res_4 = pooled_res if pooled_success else None
fe_res_4 = fe_res if fe_success else None
re_res_4 = re_res if re_success else None


In [ ]:
# ===== ТЕСТЫ СПЕЦИФИКАЦИИ =====
print("\n" + "="*70)
print("ТЕСТЫ СПЕЦИФИКАЦИИ")
print("="*70)


run_spec_tests(
    y, X, pooled_res, fe_res, re_res, pooled_success, fe_success, re_success
)


In [ ]:
# Добавляем результаты в сводную таблицу
aggregator.add_model_results(
    fe_res, 
    dependent_variable='d_Int_Rate_ConsCred_adj',
    subsample_name='Кластер_1',
    model_type='FE',
    specification_name='Модель_5'
)

In [ ]:
###############################
# Построение линейных моделей на панельных данных (кластер 2)
# (d_Int_Rate_ConsCred зависимая переменная)
###############################
print("="*70)
print("ПАНЕЛЬНАЯ РЕГРЕССИЯ: POOL + FE + RE (КЛАСТЕР 2)")
print("Зависимая переменная: d_Int_Rate_ConsCred_adj")
print("="*70)

exog_vars_initial = [
    'd_Int_Rate_ConsCred_lag1_adj',
    'd_Cred_nagr_adj',
    'd_D_top5_rozn_adj',
    'd_Fin_Dostup_adj',
    'Credit_impulse_adj',
    'd_Cred_structure_adj',
    'd_Def_Zadolg_ConsCred_adj',
    #'d_ln_New_Loans_Progr',
    'Mon_Shock',
    'd_ln_New_Loans_ConsCred_adj',
    'd_Bonds_Rate_Correct_5Y_adj',
    'd_Exc_rate_adj',
    'Inflation_Expectations_adj'                          
]

dependent_var = 'd_Int_Rate_ConsCred_adj'

# Создаем копию df_reg для работы
df_clean = df_reg_clus_two.copy()  # Ð»Ð¾Ð³Ð¸ÐºÐ° Ð±ÐµÐ· Ð¸Ð·Ð¼ÐµÐ½ÐµÐ½Ð¸Ð¹

# Приводим df_reg к формату с Region и Date как столбцами
if isinstance(df_reg.index, pd.MultiIndex):
    df_reg_temp = df_reg.reset_index()
else:
    df_reg_temp = df_reg.copy()

# Проверяем наличие зависимой переменной

if dependent_var in df_reg_temp.columns:
    
    # Присоединяем зависимую переменную по Region и Date
    df_clean = df_clean.merge(
        df_reg_temp[['Region', 'Date', dependent_var]],
        on=['Region', 'Date'],
        how='left',
        suffixes=('', '_from_reg')
    )
else:
    print(f"ERROR - {dependent_var} отсутствует в df_reg_temp")

# Проверяем независимые переменные
exog_vars_for_regression = []
for var in exog_vars_initial:
    if var in df_clean.columns:
        exog_vars_for_regression.append(var)
    else:
        print(f"  WARNING: {var} исключена")

# Удаляем NaN
cols_to_check = [dependent_var] + exog_vars_for_regression
df_clean = df_clean.dropna(subset=cols_to_check)


# ===== Установка панельного индекса =====
df_clean = df_clean.set_index(['Region', 'Date']).sort_index()

# ===== Подготовка Y и X =====
y = df_clean[[dependent_var]]
X = df_clean[exog_vars_for_regression]


# ===== POOLED OLS =====
pooled_res, fe_res, re_res, pooled_success, fe_success, re_success = run_panel_regressions(
    y, X, cov_type='robust', cluster_entity=None
)


In [ ]:
# Сохраняем результаты модели для экспорта
pooled_res_5 = pooled_res if pooled_success else None
fe_res_5 = fe_res if fe_success else None
re_res_5 = re_res if re_success else None


In [ ]:
# ===== ТЕСТЫ СПЕЦИФИКАЦИИ =====
print("\n" + "="*70)
print("ТЕСТЫ СПЕЦИФИКАЦИИ")
print("="*70)


run_spec_tests(
    y, X, pooled_res, fe_res, re_res, pooled_success, fe_success, re_success
)


In [ ]:
# Добавляем результаты в сводную таблицу
aggregator.add_model_results(
    re_res, 
    dependent_variable='d_Int_Rate_ConsCred_adj',
    subsample_name='Кластер_2',
    model_type='RE',
    specification_name='Модель_8'
)

In [ ]:
###############################
# Построение линейных моделей на панельных данных (кластер 3)
# (d_Int_Rate_ConsCred зависимая переменная)
###############################
print("="*70)
print("ПАНЕЛЬНАЯ РЕГРЕССИЯ: POOL + FE + RE (КЛАСТЕР 3)")
print("Зависимая переменная: d_Int_Rate_ConsCred_adj")
print("="*70)

exog_vars_initial = [
    'd_Int_Rate_ConsCred_lag1_adj',
    'd_Cred_nagr_adj',
    'd_D_top5_rozn_adj',
    'd_Fin_Dostup_adj',
    'Credit_impulse_adj',
    'd_Cred_structure_adj',
    'd_Def_Zadolg_ConsCred_adj',
    #'d_ln_New_Loans_Progr',
    'Mon_Shock',
    'd_ln_New_Loans_ConsCred_adj',
    'd_Bonds_Rate_Correct_5Y_adj',
    'd_Exc_rate_adj',
    'Inflation_Expectations_adj'                             
]

dependent_var = 'd_Int_Rate_ConsCred_adj'

# Создаем копию df_reg для работы
df_clean = df_reg_clus_three.copy()  # Ð»Ð¾Ð³Ð¸ÐºÐ° Ð±ÐµÐ· Ð¸Ð·Ð¼ÐµÐ½ÐµÐ½Ð¸Ð¹

# Приводим df_reg к формату с Region и Date как столбцами
if isinstance(df_reg.index, pd.MultiIndex):
    df_reg_temp = df_reg.reset_index()
else:
    df_reg_temp = df_reg.copy()

# Проверяем наличие зависимой переменной

if dependent_var in df_reg_temp.columns:
    
    # Присоединяем зависимую переменную по Region и Date
    df_clean = df_clean.merge(
        df_reg_temp[['Region', 'Date', dependent_var]],
        on=['Region', 'Date'],
        how='left',
        suffixes=('', '_from_reg')
    )
else:
    print(f"ERROR - {dependent_var} отсутствует в df_reg_temp")

# Проверяем независимые переменные
exog_vars_for_regression = []
for var in exog_vars_initial:
    if var in df_clean.columns:
        exog_vars_for_regression.append(var)
    else:
        print(f"  WARNING: {var} исключена")

# Удаляем NaN
cols_to_check = [dependent_var] + exog_vars_for_regression
df_clean = df_clean.dropna(subset=cols_to_check)


# ===== Установка панельного индекса =====
df_clean = df_clean.set_index(['Region', 'Date']).sort_index()

# ===== Подготовка Y и X =====
y = df_clean[[dependent_var]]
X = df_clean[exog_vars_for_regression]


# ===== POOLED OLS =====
pooled_res, fe_res, re_res, pooled_success, fe_success, re_success = run_panel_regressions(
    y, X, cov_type='robust', cluster_entity=None
)


In [ ]:
# Сохраняем результаты модели для экспорта
pooled_res_6 = pooled_res if pooled_success else None
fe_res_6 = fe_res if fe_success else None
re_res_6 = re_res if re_success else None


In [ ]:
# ===== ТЕСТЫ СПЕЦИФИКАЦИИ =====
print("\n" + "="*70)
print("ТЕСТЫ СПЕЦИФИКАЦИИ")
print("="*70)


run_spec_tests(
    y, X, pooled_res, fe_res, re_res, pooled_success, fe_success, re_success
)


In [ ]:
# Добавляем результаты в сводную таблицу
aggregator.add_model_results(
    re_res, 
    dependent_variable='d_Int_Rate_ConsCred_adj',
    subsample_name='Кластер_3',
    model_type='RE',
    specification_name='Модель_11'
)

In [ ]:
###############################
# Построение линейных моделей на панельных данных (общая выборка) ROISFIX
# (d_Int_Rate_ConsCred зависимая переменная)
###############################
print("="*70)
print("ПАНЕЛЬНАЯ РЕГРЕССИЯ: POOL + FE + RE (ОБЩАЯ ВЫБОРКА ROISFIX)")
print("Зависимая переменная: d_Int_Rate_ConsCred_adj")
print("="*70)

exog_vars_initial = [
    'd_Int_Rate_ConsCred_lag1_adj',
    'd_Cred_nagr_adj',
    'd_D_top5_rozn_adj',
    'd_Fin_Dostup_adj',
    'Credit_impulse_adj',
    'd_Cred_structure_adj',
    'd_Def_Zadolg_ConsCred_adj',
    #'d_ln_New_Loans_Progr',
    'd_ln_ROISFIX',
    'd_ln_New_Loans_ConsCred_adj',
    'd_Bonds_Rate_Correct_5Y_adj',
    'd_Exc_rate_adj',
    'Inflation_Expectations_adj'                          
]

dependent_var = 'd_Int_Rate_ConsCred_adj'

# Создаем копию df_reg для работы
df_clean = df_reg.copy()  # Ð»Ð¾Ð³Ð¸ÐºÐ° Ð±ÐµÐ· Ð¸Ð·Ð¼ÐµÐ½ÐµÐ½Ð¸Ð¹

# Приводим df_reg к формату с Region и Date как столбцами
if isinstance(df_reg.index, pd.MultiIndex):
    df_reg_temp = df_reg.reset_index()
else:
    df_reg_temp = df_reg.copy()

# Проверяем наличие зависимой переменной

if dependent_var in df_reg_temp.columns:
    
    # Присоединяем зависимую переменную по Region и Date
    df_clean = df_clean.merge(
        df_reg_temp[['Region', 'Date', dependent_var]],
        on=['Region', 'Date'],
        how='left',
        suffixes=('', '_from_reg')
    )
else:
    print(f"ERROR - {dependent_var} отсутствует в df_reg_temp")

# Проверяем независимые переменные
exog_vars_for_regression = []
for var in exog_vars_initial:
    if var in df_clean.columns:
        exog_vars_for_regression.append(var)
    else:
        print(f"  WARNING: {var} исключена")

# Удаляем NaN
cols_to_check = [dependent_var] + exog_vars_for_regression
df_clean = df_clean.dropna(subset=cols_to_check)


# ===== Установка панельного индекса =====
df_clean = df_clean.set_index(['Region', 'Date']).sort_index()

# ===== Подготовка Y и X =====
y = df_clean[[dependent_var]]
X = df_clean[exog_vars_for_regression]


# ===== POOLED OLS =====
pooled_res, fe_res, re_res, pooled_success, fe_success, re_success = run_panel_regressions(
    y, X, cov_type='clustered', cluster_entity=True
)


In [ ]:
# Сохраняем результаты модели для экспорта
pooled_res_7 = pooled_res if pooled_success else None
fe_res_7 = fe_res if fe_success else None
re_res_7 = re_res if re_success else None


In [ ]:
# ===== ТЕСТЫ СПЕЦИФИКАЦИИ =====
from scipy.stats import f

print("\n" + "="*70)
print("ТЕСТЫ СПЕЦИФИКАЦИИ")
print("="*70)


run_spec_tests(
    y, X, pooled_res, fe_res, re_res, pooled_success, fe_success, re_success
)


In [ ]:
# Добавляем результаты в сводную таблицу
aggregator_roisfix.add_model_results(
    re_res,
    dependent_variable='d_Int_Rate_ConsCred_adj',
    subsample_name='Общая выборка',
    model_type='RE',
    specification_name='Модель_14'
)

In [ ]:
###############################
# Построение линейных моделей на панельных данных (кластер 1 ROISFIX)
# (d_Int_Rate_ConsCred зависимая переменная)
###############################
print("="*70)
print("ПАНЕЛЬНАЯ РЕГРЕССИЯ: POOL + FE + RE (КЛАСТЕР 1 ROISFIX)")
print("Зависимая переменная: d_Int_Rate_ConsCred_adj")
print("="*70)

exog_vars_initial = [
    'd_Int_Rate_ConsCred_lag1_adj',
    'd_Cred_nagr_adj',
    'd_D_top5_rozn_adj',
    'd_Fin_Dostup_adj',
    'Credit_impulse_adj',
    'd_Cred_structure_adj',
    'd_Def_Zadolg_ConsCred_adj',
    #'d_ln_New_Loans_Progr',
    'd_ln_ROISFIX',
    'd_ln_New_Loans_ConsCred_adj',
    'd_Bonds_Rate_Correct_5Y_adj',
    'd_Exc_rate_adj',
    'Inflation_Expectations_adj'                          
]

dependent_var = 'd_Int_Rate_ConsCred_adj'

# Создаем копию df_reg для работы
df_clean = df_reg_clus_one.copy()  # Ð»Ð¾Ð³Ð¸ÐºÐ° Ð±ÐµÐ· Ð¸Ð·Ð¼ÐµÐ½ÐµÐ½Ð¸Ð¹

# Приводим df_reg к формату с Region и Date как столбцами
if isinstance(df_reg.index, pd.MultiIndex):
    df_reg_temp = df_reg.reset_index()
else:
    df_reg_temp = df_reg.copy()

# Проверяем наличие зависимой переменной

if dependent_var in df_reg_temp.columns:
    
    # Присоединяем зависимую переменную по Region и Date
    df_clean = df_clean.merge(
        df_reg_temp[['Region', 'Date', dependent_var]],
        on=['Region', 'Date'],
        how='left',
        suffixes=('', '_from_reg')
    )
else:
    print(f"ERROR - {dependent_var} отсутствует в df_reg_temp")

# Проверяем независимые переменные
exog_vars_for_regression = []
for var in exog_vars_initial:
    if var in df_clean.columns:
        exog_vars_for_regression.append(var)
    else:
        print(f"  WARNING: {var} исключена")

# Удаляем NaN
cols_to_check = [dependent_var] + exog_vars_for_regression
df_clean = df_clean.dropna(subset=cols_to_check)


# ===== Установка панельного индекса =====
df_clean = df_clean.set_index(['Region', 'Date']).sort_index()

# ===== Подготовка Y и X =====
y = df_clean[[dependent_var]]
X = df_clean[exog_vars_for_regression]


# ===== POOLED OLS =====
pooled_res, fe_res, re_res, pooled_success, fe_success, re_success = run_panel_regressions(
    y, X, cov_type='robust', cluster_entity=None
)


In [ ]:
# Сохраняем результаты модели для экспорта
pooled_res_8 = pooled_res if pooled_success else None
fe_res_8 = fe_res if fe_success else None
re_res_8 = re_res if re_success else None


In [ ]:
# ===== ТЕСТЫ СПЕЦИФИКАЦИИ =====
print("\n" + "="*70)
print("ТЕСТЫ СПЕЦИФИКАЦИИ")
print("="*70)


run_spec_tests(
    y, X, pooled_res, fe_res, re_res, pooled_success, fe_success, re_success
)


In [ ]:
# Добавляем результаты в сводную таблицу
aggregator_roisfix.add_model_results(
    re_res, 
    dependent_variable='d_Int_Rate_ConsCred_adj',
    subsample_name='Кластер_1',
    model_type='RE',
    specification_name='Модель_17'
)

In [ ]:
###############################
# Построение линейных моделей на панельных данных (кластер 2 ROISFIX)
# (d_Int_Rate_ConsCred зависимая переменная)
###############################
print("="*70)
print("ПАНЕЛЬНАЯ РЕГРЕССИЯ: POOL + FE + RE (КЛАСТЕР 2 ROISFIX)")
print("Зависимая переменная: d_Int_Rate_ConsCred_adj")
print("="*70)

exog_vars_initial = [
    'd_Int_Rate_ConsCred_lag1_adj',
    'd_Cred_nagr_adj',
    'd_D_top5_rozn_adj',
    'd_Fin_Dostup_adj',
    'Credit_impulse_adj',
    'd_Cred_structure_adj',
    'd_Def_Zadolg_ConsCred_adj',
    #'d_ln_New_Loans_Progr',
    'd_ln_ROISFIX',
    'd_ln_New_Loans_ConsCred_adj',
    'd_Bonds_Rate_Correct_5Y_adj',
    'd_Exc_rate_adj',
    'Inflation_Expectations_adj'                          
]

dependent_var = 'd_Int_Rate_ConsCred_adj'

# Создаем копию df_reg для работы
df_clean = df_reg_clus_two.copy()  # Ð»Ð¾Ð³Ð¸ÐºÐ° Ð±ÐµÐ· Ð¸Ð·Ð¼ÐµÐ½ÐµÐ½Ð¸Ð¹

# Приводим df_reg к формату с Region и Date как столбцами
if isinstance(df_reg.index, pd.MultiIndex):
    df_reg_temp = df_reg.reset_index()
else:
    df_reg_temp = df_reg.copy()

# Проверяем наличие зависимой переменной

if dependent_var in df_reg_temp.columns:
    
    # Присоединяем зависимую переменную по Region и Date
    df_clean = df_clean.merge(
        df_reg_temp[['Region', 'Date', dependent_var]],
        on=['Region', 'Date'],
        how='left',
        suffixes=('', '_from_reg')
    )
else:
    print(f"ERROR - {dependent_var} отсутствует в df_reg_temp")

# Проверяем независимые переменные
exog_vars_for_regression = []
for var in exog_vars_initial:
    if var in df_clean.columns:
        exog_vars_for_regression.append(var)
    else:
        print(f"  WARNING: {var} исключена")

# Удаляем NaN
cols_to_check = [dependent_var] + exog_vars_for_regression
df_clean = df_clean.dropna(subset=cols_to_check)


# ===== Установка панельного индекса =====
df_clean = df_clean.set_index(['Region', 'Date']).sort_index()

# ===== Подготовка Y и X =====
y = df_clean[[dependent_var]]
X = df_clean[exog_vars_for_regression]


# ===== POOLED OLS =====
pooled_res, fe_res, re_res, pooled_success, fe_success, re_success = run_panel_regressions(
    y, X, cov_type='robust', cluster_entity=None
)


In [ ]:
# Сохраняем результаты модели для экспорта
pooled_res_9 = pooled_res if pooled_success else None
fe_res_9 = fe_res if fe_success else None
re_res_9 = re_res if re_success else None


In [ ]:
# ===== ТЕСТЫ СПЕЦИФИКАЦИИ =====
print("\n" + "="*70)
print("ТЕСТЫ СПЕЦИФИКАЦИИ")
print("="*70)


run_spec_tests(
    y, X, pooled_res, fe_res, re_res, pooled_success, fe_success, re_success
)


In [ ]:
# Добавляем результаты в сводную таблицу
aggregator_roisfix.add_model_results(
    re_res, 
    dependent_variable='d_Int_Rate_ConsCred_adj',
    subsample_name='Кластер_2',
    model_type='RE',
    specification_name='Модель_20'
)

In [ ]:
###############################
# Построение линейных моделей на панельных данных (кластер 3 ROISFIX)
# (d_Int_Rate_ConsCred зависимая переменная)
###############################
print("="*70)
print("ПАНЕЛЬНАЯ РЕГРЕССИЯ: POOL + FE + RE (КЛАСТЕР 3 ROISFIX)")
print("Зависимая переменная: d_Int_Rate_ConsCred_adj")
print("="*70)

exog_vars_initial = [
    'd_Int_Rate_ConsCred_lag1_adj',
    'd_Cred_nagr_adj',
    'd_D_top5_rozn_adj',
    'd_Fin_Dostup_adj',
    'Credit_impulse_adj',
    'd_Cred_structure_adj',
    'd_Def_Zadolg_ConsCred_adj',
    #'d_ln_New_Loans_Progr',
    'd_ln_ROISFIX',
    'd_ln_New_Loans_ConsCred_adj',
    'd_Bonds_Rate_Correct_5Y_adj',
    'd_Exc_rate_adj',
    'Inflation_Expectations_adj'                             
]

dependent_var = 'd_Int_Rate_ConsCred_adj'

# Создаем копию df_reg для работы
df_clean = df_reg_clus_three.copy()  # Ð»Ð¾Ð³Ð¸ÐºÐ° Ð±ÐµÐ· Ð¸Ð·Ð¼ÐµÐ½ÐµÐ½Ð¸Ð¹

# Приводим df_reg к формату с Region и Date как столбцами
if isinstance(df_reg.index, pd.MultiIndex):
    df_reg_temp = df_reg.reset_index()
else:
    df_reg_temp = df_reg.copy()

# Проверяем наличие зависимой переменной

if dependent_var in df_reg_temp.columns:
    
    # Присоединяем зависимую переменную по Region и Date
    df_clean = df_clean.merge(
        df_reg_temp[['Region', 'Date', dependent_var]],
        on=['Region', 'Date'],
        how='left',
        suffixes=('', '_from_reg')
    )
else:
    print(f"ERROR - {dependent_var} отсутствует в df_reg_temp")

# Проверяем независимые переменные
exog_vars_for_regression = []
for var in exog_vars_initial:
    if var in df_clean.columns:
        exog_vars_for_regression.append(var)
    else:
        print(f"  WARNING: {var} исключена")

# Удаляем NaN
cols_to_check = [dependent_var] + exog_vars_for_regression
df_clean = df_clean.dropna(subset=cols_to_check)


# ===== Установка панельного индекса =====
df_clean = df_clean.set_index(['Region', 'Date']).sort_index()

# ===== Подготовка Y и X =====
y = df_clean[[dependent_var]]
X = df_clean[exog_vars_for_regression]


# ===== POOLED OLS =====
pooled_res, fe_res, re_res, pooled_success, fe_success, re_success = run_panel_regressions(
    y, X, cov_type='robust', cluster_entity=None
)


In [ ]:
# Сохраняем результаты модели для экспорта
pooled_res_10 = pooled_res if pooled_success else None
fe_res_10 = fe_res if fe_success else None
re_res_10 = re_res if re_success else None


In [ ]:
# ===== ТЕСТЫ СПЕЦИФИКАЦИИ =====
print("\n" + "="*70)
print("ТЕСТЫ СПЕЦИФИКАЦИИ")
print("="*70)


run_spec_tests(
    y, X, pooled_res, fe_res, re_res, pooled_success, fe_success, re_success
)


In [ ]:
# Добавляем результаты в сводную таблицу
aggregator_roisfix.add_model_results(
    re_res, 
    dependent_variable='d_Int_Rate_ConsCred_adj',
    subsample_name='Кластер_3',
    model_type='RE',
    specification_name='Модель_23'
)

In [ ]:
from model_results_export import ModelResultsAggregator, ensure_results_dir, add_model_set, build_and_export
import os

dep_var_name = 'd_Int_Rate_ConsCred_adj'
base_name = dep_var_name
if base_name.startswith('d_'):
    base_name = base_name[2:]
if base_name.endswith('_adj'):
    base_name = base_name[:-4]

results_dir = ensure_results_dir('Results')

model_specs_all = [
    {
        'spec_name': 'Модель_1',
        'dependent_var': dep_var_name,
        'subsample': 'Общая выборка',
        'results': {
            'pooled': pooled_res_1,
            'fe': fe_res_1,
            're': re_res_1
        }
    },
    {
        'spec_name': 'Модель_2',
        'dependent_var': dep_var_name,
        'subsample': 'Общая выборка',
        'results': {
            'pooled': pooled_res_2,
            'fe': fe_res_2,
            're': re_res_2
        }
    },
    {
        'spec_name': 'Модель_3',
        'dependent_var': dep_var_name,
        'subsample': 'Общая выборка',
        'results': {
            'pooled': pooled_res_3,
            'fe': fe_res_3,
            're': re_res_3
        }
    },
    {
        'spec_name': 'Модель_4',
        'dependent_var': dep_var_name,
        'subsample': 'Общая выборка',
        'results': {
            'pooled': pooled_res_7,
            'fe': fe_res_7,
            're': re_res_7
        }
    }
]

model_specs_cluster = [
    {
        'spec_name': 'Модель_1',
        'dependent_var': dep_var_name,
        'subsample': 'Кластер 1',
        'results': {
            'pooled': pooled_res_4,
            'fe': fe_res_4,
            're': re_res_4
        }
    },
    {
        'spec_name': 'Модель_2',
        'dependent_var': dep_var_name,
        'subsample': 'Кластер 1',
        'results': {
            'pooled': pooled_res_8,
            'fe': fe_res_8,
            're': re_res_8
        }
    },
    {
        'spec_name': 'Модель_3',
        'dependent_var': dep_var_name,
        'subsample': 'Кластер 2',
        'results': {
            'pooled': pooled_res_5,
            'fe': fe_res_5,
            're': re_res_5
        }
    },
    {
        'spec_name': 'Модель_4',
        'dependent_var': dep_var_name,
        'subsample': 'Кластер 2',
        'results': {
            'pooled': pooled_res_9,
            'fe': fe_res_9,
            're': re_res_9
        }
    },
    {
        'spec_name': 'Модель_5',
        'dependent_var': dep_var_name,
        'subsample': 'Кластер 3',
        'results': {
            'pooled': pooled_res_6,
            'fe': fe_res_6,
            're': re_res_6
        }
    },
    {
        'spec_name': 'Модель_6',
        'dependent_var': dep_var_name,
        'subsample': 'Кластер 3',
        'results': {
            'pooled': pooled_res_10,
            'fe': fe_res_10,
            're': re_res_10
        }
    }
]

aggregator_all = ModelResultsAggregator()
for spec in model_specs_all:
    add_model_set(aggregator_all, spec)

out_all = os.path.join(results_dir, f"{base_name}_all_data.xlsx")
build_and_export(aggregator_all, out_all, include_pvalues=True, decimals=3)

aggregator_cluster = ModelResultsAggregator()
for spec in model_specs_cluster:
    add_model_set(aggregator_cluster, spec)

out_cluster = os.path.join(results_dir, f"{base_name}_cluster_data.xlsx")
build_and_export(aggregator_cluster, out_cluster, include_pvalues=True, decimals=3)
